# Module 02 — Messages & Prompts

Until now you sent the model a plain string. This module is about **control**: telling the model *who it is* and *how to answer*, and writing prompts you can **reuse** with different inputs.

Two new ideas:
- **Messages** — instead of one string, you send a list of typed messages (`system`, `human`, `ai`).
- **Prompt templates** — a prompt written once with `{placeholders}` you fill in later.

Run each code cell with **Shift + Enter**, top to bottom.

## 0. Setup

Same as before: load the key, create the model.

In [1]:
import os
from dotenv import load_dotenv

load_dotenv(dotenv_path=os.path.join("..", ".env"))
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "False"

from langchain.chat_models import init_chat_model
model = init_chat_model("google_genai:gemini-2.5-flash")

print("Key loaded:", bool(os.getenv("GOOGLE_API_KEY")))

Key loaded: True


## 1. The three message types

A chat model doesn't really take a string — it takes a **list of messages**, each with a **role**:

- **`system`** — instructions *from you, the developer*: who the model is, its rules, its output style. The user never sees this. It's the most powerful lever you have.
- **`human`** — what the user says (your actual question or input).
- **`ai`** — what the model said. You include past `ai` messages to give it the conversation so far (this is the seed of "memory" in Module 07).

When you did `model.invoke("hello")` in Module 00, LangChain quietly wrapped `"hello"` into a single **human** message for you.

## 2. Your first system message

Watch how one `system` message changes everything. We tell the model to be a grumpy pirate, then ask a normal question.

The simplest way to write messages is a list of `(role, text)` tuples.

In [2]:
messages = [
    ("system", "You are a grumpy pirate. Answer everything in pirate slang, in one sentence."),
    ("human", "What is the weather like today?"),
]

response = model.invoke(messages)
print(response.text)

Arrr, the sky be lookin' like a dead man's eye, gray and full o'misery!


## 3. The same question, a different system message

Only the **system** message changed — the human question is identical. This is the whole point: the system message steers behavior while the user input stays the same.

You can also write messages as explicit objects (`SystemMessage`, `HumanMessage`) if you prefer — it's the same thing as the tuples above, just more verbose.

In [3]:
from langchain_core.messages import SystemMessage, HumanMessage

messages = [
    SystemMessage("You are a precise science teacher. Answer in one clear sentence, no jokes."),
    HumanMessage("What is the weather like today?"),
]

response = model.invoke(messages)
print(response.text)

I am an AI and do not have access to real-time, location-specific weather information.


## 4. The problem: prompts that change

Imagine you want to translate many sentences. The instruction is always the same ("translate to French"), only the sentence changes. Writing the full message list every time, by hand, is repetitive and error-prone:

```python
model.invoke([("system", "Translate to French."), ("human", "Good morning")])
model.invoke([("system", "Translate to French."), ("human", "Thank you")])
# ...copy-paste forever
```

What we want: write the prompt **once**, with blanks, and fill the blanks each time. That's a **prompt template**.

## 5. `ChatPromptTemplate` — write once, reuse

`ChatPromptTemplate.from_messages` builds a reusable prompt. Use `{curly_braces}` for the parts that change. Later you call `.invoke({...})` with a dict to fill them in.

Below, both the **language** and the **sentence** are variables, so one template handles any translation.

In [4]:
from langchain_core.prompts import ChatPromptTemplate

template = ChatPromptTemplate.from_messages([
    ("system", "You are a translator. Translate the user's text into {language}. Reply with only the translation."),
    ("human", "{text}"),
])

# Fill the blanks -> this produces the finished list of messages.
filled = template.invoke({"language": "French", "text": "Good morning, how are you?"})
print(filled.messages)

[SystemMessage(content="You are a translator. Translate the user's text into French. Reply with only the translation.", additional_kwargs={}, response_metadata={}), HumanMessage(content='Good morning, how are you?', additional_kwargs={}, response_metadata={})]


## 6. Template + model together

A template just *builds messages*; it doesn't call the model. So you do two steps: fill the template, then `invoke` the model with the result.

Run it with different inputs and notice you never rewrote the prompt — you only changed the dict.

In [5]:
def translate(language, text):
    prompt = template.invoke({"language": language, "text": text})
    return model.invoke(prompt).text

print(translate("French", "Where is the train station?"))
print(translate("Japanese", "Where is the train station?"))
print(translate("German", "I would like a coffee, please."))

Où est la gare ?
駅はどこですか？
Ich hätte gerne einen Kaffee, bitte.


## Recap

- A chat model takes a **list of messages**, each with a role: **`system`**, **`human`**, **`ai`**.
- The **system message** sets persona, rules, and output style — your strongest control lever.
- Write messages as `(role, text)` tuples or as `SystemMessage` / `HumanMessage` objects — same result.
- A **`ChatPromptTemplate`** is a prompt written once with `{placeholders}`; fill it with `.invoke({...})`.
- Template builds messages → model invokes them. Reuse the template, change only the inputs.

## 🛠️ Try it yourself

1. Change the system message in section 2 to a persona you like (a chef, a lawyer, Yoda). Keep the human question the same and watch the style shift.
2. Add a rule to a system message — e.g. "always answer in exactly 5 words" — and confirm the model obeys.
3. Build your own `ChatPromptTemplate` with **one** variable (e.g. "Write a haiku about {topic}.") and run it for three topics.
4. Add a second variable to your template (e.g. `{tone}`) and fill both.
5. **Stretch:** put an `ai` message in a message list (a fake previous answer) before a new `human` message, and see the model treat it as conversation history.

When you're done, say **"next"** and we'll build **Module 03 — Prompt & Context Engineering**.